# HyDE Pipeline: Query Expansion + MiniLM + Cross-Encoder Reranker
## Hypothetical Document Embedding for improved company retrieval

**What this notebook does:**

Implements and evaluates two progressive pipeline stages on top of the
MiniLM all-fields baseline:

**Stage 1 — HyDE + MiniLM:**
Instead of encoding the raw query, a small LLM generates an ideal company
description for the query. MiniLM then encodes that rich description and
searches the FAISS index. This gives MiniLM a much more informative query
vector to work with.

**Stage 2 — HyDE + MiniLM + Cross-Encoder Reranker:**
The top-50 candidates from Stage 1 are re-scored by a cross-encoder that
reads the original query AND each company description together — like a human
would — giving a much more precise relevance score.

**Why this order makes sense:**
```
98,716 companies
      ↓ HyDE + MiniLM (semantic search on expanded query)
Top 1000 candidates  ← evaluated here (Stage 1)
      ↓ Cross-encoder (reads query + each of top-50 together)
Re-ranked top 50     ← evaluated here (Stage 2)
      ↓
Positions 51-1000 stay as MiniLM ranked them
```

The reranker CANNOT fix what MiniLM missed — if a relevant company is
not in the top 1000, the reranker never sees it. This is why Stage 1
recall matters so much, and why HyDE's query expansion is important:
a richer query → better MiniLM recall → better reranker input.

**LLM used for HyDE:** `llama3.1:8b` via Ollama (local, no API cost)
**Embedder:** `all-MiniLM-L6-v2` all fields (best performing baseline)
**Reranker:** `BAAI/bge-reranker-v2-m3`

**Folder structure:**
```
result/
└── 06_hyde_pipeline/
    ├── hyde_queries.json              # LLM-generated descriptions per query
    ├── hyde_minilm_results.csv        # Stage 1: top-1000 per query
    ├── hyde_reranked_results.csv      # Stage 2: top-50 re-ranked per query
    ├── evaluation_stage1.csv          # NDCG/Prec/Recall/F1 for HyDE+MiniLM
    ├── evaluation_stage2.csv          # NDCG/Prec/Recall/F1 for HyDE+MiniLM+Reranker
    ├── latency_breakdown.csv          # Per-query timing for all 3 components
    └── comparison_all.csv             # vs MiniLM baseline
```

### Notebook structure
1. Environment setup
2. Imports
3. Load corpus, embeddings & queries
4. Load models (MiniLM + Reranker)
5. HyDE — generate ideal company descriptions for all 101 queries
6. Inspect generated descriptions (quality check)
7. Stage 1 — HyDE + MiniLM retrieval
8. Stage 1 evaluation — NDCG, Precision, Recall, F1
9. Stage 2 — Cross-encoder reranker on top-50
10. Stage 2 evaluation — NDCG, Precision, Recall, F1
11. Latency breakdown
12. Final comparison table
13. Key findings

## 1 · Environment Setup

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

RESULT_DIR = Path('result/06_hyde_pipeline')
RESULT_DIR.mkdir(parents=True, exist_ok=True)
print(f'[Setup] Result folder : {RESULT_DIR}/ — ready')

# ── Ollama settings ───────────────────────────────────────────────────────────
OLLAMA_BASE_URL = os.getenv('OLLAMA_BASE_URL', 'http://localhost:11434')
HYDE_MODEL      = 'llama3.1:8b'   # change if your Ollama uses a different name
print(f'[Setup] Ollama URL    : {OLLAMA_BASE_URL}')
print(f'[Setup] HyDE model    : {HYDE_MODEL}')

## 2 · Imports

| Package | Role |
|---|---|
| `requests` | Call local Ollama API for HyDE generation |
| `sentence_transformers` | MiniLM encoder |
| `transformers` | BGE cross-encoder reranker |
| `faiss` | Vector similarity search |
| `torch` | GPU detection |

In [ ]:
import json, time, requests
import numpy as np
import pandas as pd
import faiss
import torch
from pathlib import Path
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForSequenceClassification, AutoTokenizer

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'[Imports] Device      : {DEVICE}')
if torch.cuda.is_available():
    print(f'[Imports] GPU         : {torch.cuda.get_device_name(0)}')
    print(f'[Imports] VRAM        : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
print('[Imports] All packages loaded successfully')

## 3 · Load Corpus, Embeddings & Queries

We reuse:
- `result/03_baseline_minilm/company_embeddings.npy` — pre-computed MiniLM all-fields embeddings
- `dataset/production_results.xlsx` — company corpus and relevance labels
- `dataset/goi_search_results.json` — 101 queries

No re-encoding needed — embeddings are already computed.

In [ ]:
print('[Load] Loading corpus...')
results_df    = pd.read_excel('dataset/production_results.xlsx')
all_companies = results_df.drop_duplicates(subset='domain').reset_index(drop=True)
print(f'[Load] Unique companies  : {len(all_companies):,}')

print('[Load] Loading queries...')
with open('dataset/goi_search_results.json', 'r') as f:
    data = json.load(f)
print(f'[Load] Queries           : {len(data)}')

print('[Load] Loading production labels...')
production_df = pd.read_excel('dataset/production_results.xlsx')
print(f'[Load] Production rows   : {len(production_df):,}')

# ── Load pre-computed MiniLM all-fields embeddings ────────────────────────────
print('[Load] Loading MiniLM embeddings...')
embeddings = np.load('result/03_baseline_minilm/company_embeddings.npy').astype('float32')
print(f'[Load] Embeddings shape  : {embeddings.shape}')  # (98716, 384)

# ── Build FAISS index from pre-computed embeddings ───────────────────────────
print('[Load] Building FAISS index...')
t0    = time.time()
index = faiss.IndexFlatL2(embeddings.shape[1])  # L2 for MiniLM (not normalised)
index.add(embeddings)
print(f'[Load] FAISS index built : {index.ntotal:,} vectors in {time.time()-t0:.2f}s')

## 4 · Load MiniLM + Cross-Encoder Reranker

Both models loaded once and reused for all 101 queries.

**MiniLM** — encodes the HyDE-generated company descriptions at query time.
**BGE Reranker** — scores (query, company_summary) pairs for Stage 2.

In [ ]:
# ── MiniLM ───────────────────────────────────────────────────────────────────
print('[Models] Loading MiniLM...')
t0          = time.time()
model_minilm = SentenceTransformer('all-MiniLM-L6-v2', device=DEVICE)
print(f'[Models] MiniLM loaded in  : {time.time()-t0:.1f}s on {model_minilm.device}')

# ── BGE Cross-Encoder Reranker ────────────────────────────────────────────────
print('[Models] Loading BGE reranker...')
print('[Models] First run downloads ~570MB...')
t0                 = time.time()
RERANKER_MODEL     = 'BAAI/bge-reranker-v2-m3'
tokenizer_reranker = AutoTokenizer.from_pretrained(RERANKER_MODEL)
model_reranker     = AutoModelForSequenceClassification.from_pretrained(RERANKER_MODEL)
model_reranker     = model_reranker.to(DEVICE)
model_reranker.eval()
print(f'[Models] Reranker loaded in : {time.time()-t0:.1f}s on {DEVICE}')

# ── Ollama connectivity check ─────────────────────────────────────────────────
print('[Models] Checking Ollama...')
try:
    resp      = requests.get(f'{OLLAMA_BASE_URL}/api/tags', timeout=5)
    available = [m['name'] for m in resp.json().get('models', [])]
    print(f'[Models] Ollama running    : {OLLAMA_BASE_URL}')
    print(f'[Models] Available models  : {available}')
    if HYDE_MODEL not in available:
        print(f'[Models] WARNING: {HYDE_MODEL} not found — run: ollama pull {HYDE_MODEL}')
    else:
        print(f'[Models] HyDE model ready  : {HYDE_MODEL} ✅')
except Exception as e:
    print(f'[Models] ERROR: Cannot reach Ollama — {e}')
    print('[Models] Start Ollama with: ollama serve')

## 5 · HyDE — Generate Ideal Company Descriptions

**What HyDE does (plain English):**
Instead of embedding the short query `"software companies in Germany"`,
we ask an LLM to write what a perfect matching company would look like.
MiniLM then embeds that rich description instead.

**Why this works:**
- The generated description is in the same "language" as company summaries
- It expands the query with relevant synonyms and related terms automatically
- A description of an ideal software company will naturally mention
  "programming", "SaaS", "applications" — matching real company summaries better

**System prompt design:**
- Instructs the LLM to write as if describing a real company
- Keeps it to 2-3 sentences — long enough to be informative, short enough to encode well
- Temperature=0 for deterministic, reproducible results

**Saved to:** `result/06_hyde_pipeline/hyde_queries.json`
so you can reload without re-running the LLM.

In [ ]:
HYDE_SYSTEM_PROMPT = """You are helping search a company database.
Given a search query, write a company profile in this EXACT format:

Company: [company name placeholder] | Country: [country if mentioned, else omit] | 
State: [state if mentioned, else omit] | Industry: NACE [letter]: [description] | 
[2 sentence natural description of what the company does] | 
Keywords: [5-8 relevant keywords separated by commas]

Output only the profile, nothing else."""

def generate_hyde_description(query, max_retries=3):
    """Generate an ideal company description for a query using local Ollama LLM."""
    for attempt in range(max_retries):
        try:
            response = requests.post(
                f'{OLLAMA_BASE_URL}/api/chat',
                json={
                    'model':   HYDE_MODEL,
                    'messages': [
                        {'role': 'system', 'content': HYDE_SYSTEM_PROMPT},
                        {'role': 'user',   'content': f'Query: {query}'}
                    ],
                    'stream':  False,
                    'options': {'temperature': 0.0, 'num_predict': 150}
                },
                timeout=60
            )
            response.raise_for_status()
            description = response.json()['message']['content'].strip()
            return description
        except Exception as e:
            if attempt == max_retries - 1:
                print(f'  [HyDE] WARNING: Failed for "{query}": {e}')
                return query  # fallback to original query
            time.sleep(1)

# ── Check if HyDE descriptions already exist ──────────────────────────────────
hyde_path = RESULT_DIR / 'hyde_queries.json'

if hyde_path.exists():
    print('[HyDE] Loading existing descriptions from disk...')
    with open(hyde_path) as f:
        hyde_results = json.load(f)
    print(f'[HyDE] Loaded {len(hyde_results)} descriptions')

else:
    print(f'[HyDE] Generating descriptions for {len(data)} queries...')
    print(f'[HyDE] Model: {HYDE_MODEL} | Temp: 0.0 | Max tokens: 150')
    print('-' * 55)

    hyde_results = []
    total_start  = time.time()
    gen_times    = []

    for i, item in enumerate(data):
        qid   = item['query_id']
        query = item['query']

        t0          = time.perf_counter()
        description = generate_hyde_description(query)
        gen_ms      = (time.perf_counter() - t0) * 1000
        gen_times.append(gen_ms)

        hyde_results.append({
            'query_id':    qid,
            'query':       query,
            'hyde_desc':   description,
            'gen_ms':      round(gen_ms, 1),
        })

        if (i + 1) % 10 == 0 or (i + 1) == len(data):
            elapsed   = time.time() - total_start
            remaining = (len(data) - i - 1) * elapsed / (i + 1)
            print(f'[HyDE] {i+1:3d}/{len(data)}  |  '
                  f'avg {sum(gen_times)/len(gen_times):.0f}ms/query  |  '
                  f'~{remaining:.0f}s remaining')

    with open(hyde_path, 'w') as f:
        json.dump(hyde_results, f, indent=2)

    avg_gen = sum(gen_times) / len(gen_times)
    print('-' * 55)
    print(f'[HyDE] Done! Avg generation time : {avg_gen:.0f}ms/query')
    print(f'[HyDE] Saved to                  : {hyde_path}')

## 6 · Inspect HyDE Quality

Before running retrieval, check what the LLM actually generated.
This is the most important sanity check — if the descriptions are generic
or off-topic, HyDE will not help.

**What to look for:**
- Does the description match the intent of the query?
- Does it contain relevant industry terms beyond what the query said?
- Is it specific enough to distinguish relevant from irrelevant companies?

In [ ]:
print('[Inspect] === HYDE DESCRIPTION QUALITY CHECK ===')
print()

# Show 10 examples — first few and some middle ones
sample_indices = [0, 1, 2, 5, 10, 15, 20, 30, 50, 75]
for idx in sample_indices:
    if idx < len(hyde_results):
        item = hyde_results[idx]
        print(f'Query   : "{item["query"]}"')
        print(f'HyDE    : {item["hyde_desc"]}')
        print(f'Gen time: {item["gen_ms"]:.0f}ms')
        print('-' * 70)

# Stats
avg_len = sum(len(r['hyde_desc']) for r in hyde_results) / len(hyde_results)
fallbacks = sum(1 for r in hyde_results if r['hyde_desc'] == r['query'])
print(f'\n[Inspect] Avg description length : {avg_len:.0f} chars')
print(f'[Inspect] Fallbacks (used raw query): {fallbacks}/{len(hyde_results)}')

## 7 · Stage 1 — HyDE + MiniLM Retrieval

Encode each HyDE description with MiniLM and search the FAISS index.

**Key difference from baseline:**
- Baseline: MiniLM encodes `"software companies in Germany"` (7 words)
- HyDE: MiniLM encodes a 2-3 sentence ideal company description (50-80 words)

The longer, richer description gives MiniLM much more signal to work with.

**Output:** `result/06_hyde_pipeline/hyde_minilm_results.csv`

In [ ]:
# ── GPU warmup before measuring latency ──────────────────────────────────────
print('[Warmup] Warming up GPU with 5 dummy queries...')
warmup_text = hyde_results[0]['hyde_desc']
for _ in range(5):
    dummy_emb = model_minilm.encode(
        [warmup_text], convert_to_numpy=True
    ).astype('float32')
    index.search(dummy_emb, 1000)
print('[Warmup] GPU warm — latency measurement starting now')
print()

print(f'[Stage1] Starting HyDE + MiniLM retrieval for {len(data)} queries...')
print(f'[Stage1] Encoding HyDE descriptions (not raw queries)')
print('-' * 60)

all_stage1_results = []
stage1_times       = []   # encode + search time only (not HyDE gen)
total_start        = time.time()

for i, item in enumerate(hyde_results):
    qid        = item['query_id']
    query      = item['query']
    hyde_desc  = item['hyde_desc']

    # ── Encode HyDE description + search ─────────────────────────────────────
    t0       = time.perf_counter()
    q_emb    = model_minilm.encode(
        [hyde_desc],
        normalize_embeddings=False,  # MiniLM uses L2
        convert_to_numpy=True
    ).astype('float32')
    distances, idxs = index.search(q_emb, 1000)
    encode_search_ms = (time.perf_counter() - t0) * 1000
    stage1_times.append(encode_search_ms)

    for rank, (idx, dist) in enumerate(zip(idxs[0], distances[0])):
        company = all_companies.iloc[idx]
        all_stage1_results.append({
            'query_id':  qid,
            'query':     query,
            'rank':      rank + 1,
            'score':     float(dist),
            'domain':    company['domain'],
            'name':      company.get('name', ''),
            'country':   company.get('country', ''),
            'summary':   company.get('summary', ''),
        })

    if (i + 1) % 20 == 0 or (i + 1) == len(data):
        elapsed   = time.time() - total_start
        remaining = (len(data) - i - 1) * elapsed / (i + 1)
        print(f'[Stage1] {i+1:3d}/{len(data)}  |  '
              f'avg {sum(stage1_times)/len(stage1_times):.1f}ms/query (encode+search)  |  '
              f'~{remaining:.0f}s remaining')

stage1_df = pd.DataFrame(all_stage1_results)
stage1_df.to_csv(RESULT_DIR / 'hyde_minilm_results.csv', index=False)

AVG_ENCODE_SEARCH_MS = sum(stage1_times) / len(stage1_times)
AVG_HYDE_GEN_MS      = sum(r['gen_ms'] for r in hyde_results) / len(hyde_results)
AVG_STAGE1_TOTAL_MS  = AVG_HYDE_GEN_MS + AVG_ENCODE_SEARCH_MS

print('-' * 60)
print(f'[Stage1] Done!')
print(f'[Stage1] Avg HyDE generation     : {AVG_HYDE_GEN_MS:.0f}ms')
print(f'[Stage1] Avg encode + search      : {AVG_ENCODE_SEARCH_MS:.1f}ms')
print(f'[Stage1] Avg total (HyDE+MiniLM)  : {AVG_STAGE1_TOTAL_MS:.0f}ms')
print(f'[Stage1] Saved to                 : result/06_hyde_pipeline/hyde_minilm_results.csv')

## 8 · Stage 1 Evaluation — HyDE + MiniLM

Same evaluation protocol as all baseline notebooks.
Pseudo-relevance: company is relevant if in production top-100.

In [ ]:
K_VALUES = [10, 50, 100, 500, 1000]

def get_relevant(query_id, top_k=100):
    return set(production_df[
        (production_df['query_id'] == query_id) &
        (production_df['rank'] <= top_k)
    ]['domain'].tolist())

def precision_at_k(retrieved, relevant, k):
    return len(set(retrieved[:k]) & relevant) / k if k else 0

def recall_at_k(retrieved, relevant, k):
    return len(set(retrieved[:k]) & relevant) / len(relevant) if relevant else 0

def f1_at_k(retrieved, relevant, k):
    p = precision_at_k(retrieved, relevant, k)
    r = recall_at_k(retrieved, relevant, k)
    return 2 * p * r / (p + r) if (p + r) > 0 else 0

def dcg_at_k(retrieved, relevant, k):
    return sum(
        1 / np.log2(i + 2)
        for i, d in enumerate(retrieved[:k]) if d in relevant
    )

def ndcg_at_k(retrieved, relevant, k):
    ideal = dcg_at_k(list(relevant), relevant, k)
    return dcg_at_k(retrieved, relevant, k) / ideal if ideal else 0

print('[Eval1] Evaluating Stage 1 (HyDE + MiniLM)...')
eval1_rows = []

for i, item in enumerate(data):
    qid       = item['query_id']
    relevant  = get_relevant(qid)
    retrieved = (
        stage1_df[stage1_df['query_id'] == qid]
        .sort_values('rank')['domain'].tolist()
    )
    for k in K_VALUES:
        eval1_rows.append({
            'query_id':  qid,
            'query':     item['query'],
            'k':         k,
            'precision': precision_at_k(retrieved, relevant, k),
            'recall':    recall_at_k(retrieved, relevant, k),
            'f1':        f1_at_k(retrieved, relevant, k),
            'ndcg':      ndcg_at_k(retrieved, relevant, k),
        })

    if (i + 1) % 25 == 0:
        print(f'[Eval1] {i+1}/101 queries evaluated...')

eval1_df = pd.DataFrame(eval1_rows)
eval1_df.to_csv(RESULT_DIR / 'evaluation_stage1.csv', index=False)

print('[Eval1] === STAGE 1 RESULTS: HyDE + MiniLM ===')
print(f'  {"k":<6} {"NDCG":>8} {"Precision":>10} {"Recall":>8} {"F1":>8}')
print('  ' + '-' * 46)
for k in K_VALUES:
    sub = eval1_df[eval1_df['k'] == k]
    print(f'  {k:<6} '
          f'{sub["ndcg"].mean():>8.3f} '
          f'{sub["precision"].mean():>10.3f} '
          f'{sub["recall"].mean():>8.3f} '
          f'{sub["f1"].mean():>8.3f}')

## 9 · Stage 2 — Cross-Encoder Reranker on top-50

Takes the top-50 from Stage 1 and re-scores each one by reading the
**original query** AND the **company summary** together.

**Why we use the original query (not the HyDE description) for reranking:**
The reranker needs to score actual relevance to the user's intent.
The HyDE description was used to find candidates — the original query
is what the user actually wants. Using the original query for reranking
gives a more accurate relevance score.

**Combined list for evaluation:**
- Positions 1-50: re-ranked by cross-encoder
- Positions 51-1000: original MiniLM order from Stage 1
This is the fairest evaluation of the full pipeline.

In [ ]:
def rerank(query, candidates_df, top_k=50, batch_size=32):
    """
    Re-rank top-k candidates using cross-encoder.
    Uses ORIGINAL query (not HyDE description) for accurate relevance scoring.
    Returns reranked dataframe.
    """
    candidates = candidates_df.head(top_k).copy()
    summaries  = candidates['summary'].fillna('').tolist()
    pairs      = [[query, s] for s in summaries]
    all_scores = []

    with torch.no_grad():
        for i in range(0, len(pairs), batch_size):
            batch   = pairs[i:i+batch_size]
            encoded = tokenizer_reranker(
                batch, padding=True, truncation=True,
                max_length=512, return_tensors='pt'
            ).to(DEVICE)
            scores = model_reranker(**encoded).logits.squeeze(-1)
            all_scores.extend(scores.cpu().float().tolist())

    candidates['reranker_score'] = all_scores
    candidates['stage1_rank']    = candidates['rank'].values
    reranked = candidates.sort_values('reranker_score', ascending=False).reset_index(drop=True)
    reranked['rank'] = reranked.index + 1
    return reranked

# ── GPU warmup for reranker ───────────────────────────────────────────────────
print('[Warmup] Warming up reranker with 3 dummy calls...')
warmup_cands = stage1_df[stage1_df['query_id'] == data[0]['query_id']].head(50)
for _ in range(3):
    rerank(data[0]['query'], warmup_cands)
print('[Warmup] Reranker warm ✅')
print()

print(f'[Stage2] Re-ranking top-50 for {len(data)} queries...')
print(f'[Stage2] Using ORIGINAL query for reranking (not HyDE description)')
print('-' * 60)

all_stage2_results = []
rerank_times       = []
total_start        = time.time()

for i, item in enumerate(data):
    qid   = item['query_id']
    query = item['query']  # ORIGINAL query — not HyDE

    # Get Stage 1 top-1000 for this query
    stage1_query = stage1_df[stage1_df['query_id'] == qid].sort_values('rank')

    # ── Rerank top-50 ─────────────────────────────────────────────────────────
    t0       = time.perf_counter()
    reranked = rerank(query, stage1_query, top_k=50)
    rerank_ms = (time.perf_counter() - t0) * 1000
    rerank_times.append(rerank_ms)

    # ── Combined: reranked top-50 + original positions 51-1000 ───────────────
    reranked_domains = set(reranked['domain'].tolist())
    remaining = stage1_query[~stage1_query['domain'].isin(reranked_domains)]

    for _, row in reranked.iterrows():
        all_stage2_results.append({
            'query_id':       qid,
            'query':          query,
            'rank':           int(row['rank']),
            'stage1_rank':    int(row['stage1_rank']),
            'reranker_score': float(row['reranker_score']),
            'domain':         row['domain'],
            'name':           row.get('name', ''),
            'summary':        row.get('summary', ''),
        })

    for rank_offset, (_, row) in enumerate(remaining.iterrows(), start=51):
        all_stage2_results.append({
            'query_id':       qid,
            'query':          query,
            'rank':           rank_offset,
            'stage1_rank':    int(row['rank']),
            'reranker_score': None,
            'domain':         row['domain'],
            'name':           row.get('name', ''),
            'summary':        row.get('summary', ''),
        })

    if (i + 1) % 20 == 0 or (i + 1) == len(data):
        elapsed   = time.time() - total_start
        remaining_time = (len(data) - i - 1) * elapsed / (i + 1)
        print(f'[Stage2] {i+1:3d}/{len(data)}  |  '
              f'avg {sum(rerank_times)/len(rerank_times):.0f}ms/query  |  '
              f'~{remaining_time:.0f}s remaining')

stage2_df = pd.DataFrame(all_stage2_results)
stage2_df.to_csv(RESULT_DIR / 'hyde_reranked_results.csv', index=False)

AVG_RERANK_MS       = sum(rerank_times) / len(rerank_times)
AVG_STAGE2_TOTAL_MS = AVG_HYDE_GEN_MS + AVG_ENCODE_SEARCH_MS + AVG_RERANK_MS

print('-' * 60)
print(f'[Stage2] Done!')
print(f'[Stage2] Avg reranker time         : {AVG_RERANK_MS:.0f}ms')
print(f'[Stage2] Avg total pipeline        : {AVG_STAGE2_TOTAL_MS:.0f}ms')
print(f'[Stage2] Saved to                  : result/06_hyde_pipeline/hyde_reranked_results.csv')

## 10 · Stage 2 Evaluation — HyDE + MiniLM + Reranker

The re-ranked list is evaluated using the same metrics.
Note: only positions 1-50 were touched by the reranker.
Improvement is expected mainly at k=10 and k=50.

In [ ]:
print('[Eval2] Evaluating Stage 2 (HyDE + MiniLM + Reranker)...')
eval2_rows = []

for i, item in enumerate(data):
    qid       = item['query_id']
    relevant  = get_relevant(qid)
    retrieved = (
        stage2_df[stage2_df['query_id'] == qid]
        .sort_values('rank')['domain'].tolist()
    )
    for k in K_VALUES:
        eval2_rows.append({
            'query_id':  qid,
            'query':     item['query'],
            'k':         k,
            'precision': precision_at_k(retrieved, relevant, k),
            'recall':    recall_at_k(retrieved, relevant, k),
            'f1':        f1_at_k(retrieved, relevant, k),
            'ndcg':      ndcg_at_k(retrieved, relevant, k),
        })

    if (i + 1) % 25 == 0:
        print(f'[Eval2] {i+1}/101 queries evaluated...')

eval2_df = pd.DataFrame(eval2_rows)
eval2_df.to_csv(RESULT_DIR / 'evaluation_stage2.csv', index=False)

print('[Eval2] === STAGE 2 RESULTS: HyDE + MiniLM + Reranker ===')
print(f'  {"k":<6} {"NDCG":>8} {"Precision":>10} {"Recall":>8} {"F1":>8}')
print('  ' + '-' * 46)
for k in K_VALUES:
    sub = eval2_df[eval2_df['k'] == k]
    print(f'  {k:<6} '
          f'{sub["ndcg"].mean():>8.3f} '
          f'{sub["precision"].mean():>10.3f} '
          f'{sub["recall"].mean():>8.3f} '
          f'{sub["f1"].mean():>8.3f}')

## 11 · Latency Breakdown

Full pipeline timing compared to MiniLM baseline (16.9ms).

**Components:**
- HyDE generation: LLM call to Ollama (includes model inference)
- MiniLM encode + search: encode HyDE description + FAISS search
- Reranker: cross-encoder on top-50 candidates

In [ ]:
MINILM_BASELINE_MS = 16.9  # from baseline experiment

print('[Latency] ============================================================')
print('[Latency] PIPELINE LATENCY BREAKDOWN')
print('[Latency] ============================================================')
print(f'\n  Component               |  Stage 1         |  Stage 2')
print(f'  ----------------------- | ---------------- | ----------------')
print(f'  MiniLM baseline         | {MINILM_BASELINE_MS:>8.1f}ms       | {MINILM_BASELINE_MS:>8.1f}ms')
print(f'  + HyDE generation       | {AVG_HYDE_GEN_MS:>8.0f}ms       | {AVG_HYDE_GEN_MS:>8.0f}ms')
print(f'  + MiniLM encode+search  | {AVG_ENCODE_SEARCH_MS:>8.1f}ms       | {AVG_ENCODE_SEARCH_MS:>8.1f}ms')
print(f'  + Cross-encoder rerank  |       N/A         | {AVG_RERANK_MS:>8.0f}ms')
print(f'  ----------------------- | ---------------- | ----------------')
print(f'  TOTAL                   | {AVG_STAGE1_TOTAL_MS:>8.0f}ms       | {AVG_STAGE2_TOTAL_MS:>8.0f}ms')
print(f'\n[Latency] Note: HyDE generation can be parallelised or cached')
print(f'[Latency] for repeated/similar queries in production.')

# Save latency
latency_df = pd.DataFrame([{
    'pipeline':          'MiniLM baseline',
    'hyde_gen_ms':       0,
    'encode_search_ms':  MINILM_BASELINE_MS,
    'rerank_ms':         0,
    'total_ms':          MINILM_BASELINE_MS,
}, {
    'pipeline':          'HyDE + MiniLM',
    'hyde_gen_ms':       round(AVG_HYDE_GEN_MS, 1),
    'encode_search_ms':  round(AVG_ENCODE_SEARCH_MS, 1),
    'rerank_ms':         0,
    'total_ms':          round(AVG_STAGE1_TOTAL_MS, 1),
}, {
    'pipeline':          'HyDE + MiniLM + Reranker',
    'hyde_gen_ms':       round(AVG_HYDE_GEN_MS, 1),
    'encode_search_ms':  round(AVG_ENCODE_SEARCH_MS, 1),
    'rerank_ms':         round(AVG_RERANK_MS, 1),
    'total_ms':          round(AVG_STAGE2_TOTAL_MS, 1),
}])
latency_df.to_csv(RESULT_DIR / 'latency_breakdown.csv', index=False)
print(f'\n[Latency] Saved to result/06_hyde_pipeline/latency_breakdown.csv')

## 12 · Final Comparison Table

All three pipelines compared at all k values.
Load MiniLM baseline from previous notebook for direct comparison.

In [ ]:
# Load MiniLM all-fields baseline
try:
    baseline_eval = pd.read_csv('result/03_baseline_minilm/evaluation_minilm.csv')
    has_baseline  = True
    print('[Compare] Loaded MiniLM baseline evaluation')
except FileNotFoundError:
    has_baseline  = False
    print('[Compare] WARNING: MiniLM baseline not found — showing HyDE results only')

print('\n[Compare] ============================================================')
print('[Compare] FULL PIPELINE COMPARISON')
print('[Compare] ============================================================')
print(f'  {"Method":<30} {"k":>6} | {"NDCG":>7} | {"Prec":>7} | {"Recall":>7} | {"F1":>7}')
print('  ' + '=' * 70)

methods = []
if has_baseline:
    methods.append(('MiniLM all fields (baseline)', baseline_eval))
methods.append(('HyDE + MiniLM',                    eval1_df))
methods.append(('HyDE + MiniLM + Reranker',         eval2_df))

comp_rows = []
for method_name, df in methods:
    for k in K_VALUES:
        sub  = df[df['k'] == k]
        ndcg = sub['ndcg'].mean()
        prec = sub['precision'].mean()
        rec  = sub['recall'].mean()
        f1   = sub['f1'].mean()
        print(f'  {method_name:<30} {k:>6} | {ndcg:>7.3f} | {prec:>7.3f} | {rec:>7.3f} | {f1:>7.3f}')
        comp_rows.append({'method':method_name,'k':k,
                          'ndcg':round(ndcg,3),'precision':round(prec,3),
                          'recall':round(rec,3),'f1':round(f1,3)})
    print('  ' + '-' * 70)

pd.DataFrame(comp_rows).to_csv(RESULT_DIR / 'comparison_all.csv', index=False)
print(f'\n[Compare] Saved to result/06_hyde_pipeline/comparison_all.csv')

## 13 · Key Findings

Auto-generated summary of improvements at each stage.

In [ ]:
print('[Findings] ============================================================')
print('[Findings] KEY FINDINGS')
print('[Findings] ============================================================')

if has_baseline:
    base_n10 = baseline_eval[baseline_eval['k']==10]['ndcg'].mean()
    base_p10 = baseline_eval[baseline_eval['k']==10]['precision'].mean()
else:
    base_n10 = base_p10 = None

s1_n10 = eval1_df[eval1_df['k']==10]['ndcg'].mean()
s1_p10 = eval1_df[eval1_df['k']==10]['precision'].mean()
s2_n10 = eval2_df[eval2_df['k']==10]['ndcg'].mean()
s2_p10 = eval2_df[eval2_df['k']==10]['precision'].mean()

print(f'\n[Findings] NDCG@10 progression:')
if base_n10:
    print(f'  MiniLM baseline          : {base_n10:.3f}')
    print(f'  + HyDE                   : {s1_n10:.3f}  ({(s1_n10-base_n10)/base_n10*100:+.1f}% vs baseline)')
    print(f'  + HyDE + Reranker        : {s2_n10:.3f}  ({(s2_n10-base_n10)/base_n10*100:+.1f}% vs baseline)')
else:
    print(f'  HyDE + MiniLM            : {s1_n10:.3f}')
    print(f'  HyDE + MiniLM + Reranker : {s2_n10:.3f}  ({(s2_n10-s1_n10)/s1_n10*100:+.1f}% vs Stage 1)')

print(f'\n[Findings] Latency progression:')
print(f'  MiniLM baseline          : {MINILM_BASELINE_MS:.1f}ms')
print(f'  HyDE + MiniLM            : {AVG_STAGE1_TOTAL_MS:.0f}ms  (+{AVG_HYDE_GEN_MS:.0f}ms HyDE generation)')
print(f'  HyDE + MiniLM + Reranker : {AVG_STAGE2_TOTAL_MS:.0f}ms  (+{AVG_RERANK_MS:.0f}ms reranking)')
print('[Findings] ============================================================')